In [1]:
"""
baseline_lda.py
---------------
Trains sklearn LDA as a baseline and evaluates:
    1. Perplexity (using same log-likelihood formulation as AVITM)
    2. Topic Coherence (NPMI, using same training-corpus method as AVITM)

Results are saved to checkpoints/lda_baseline_results.json for use in notebook.
"""

import numpy as np
import json
from pathlib import Path
from sklearn.decomposition import LatentDirichletAllocation
from itertools import combinations

from data import get_dataloaders


# ── Constants ──────────────────────────────────────────────────────────────
N_TOPICS     = 50
MAX_ITER     = 100
RANDOM_STATE = 42


# ── 1. Perplexity ──────────────────────────────────────────────────────────
def compute_lda_perplexity(lda, X, vocab_size):
    """
    Compute perplexity using log-likelihood, consistent with AVITM method.

    sklearn's lda.score(X) returns per-sample log-likelihood (sum over words).
    We convert this to per-word log-likelihood and then exponentiate.

    Parameters
    ----------
    lda       : fitted LatentDirichletAllocation
    X         : sparse BOW matrix (n_docs, vocab_size)
    vocab_size : int

    Returns
    -------
    perplexity : float
    """
    # score() returns total log-likelihood over all documents
    total_log_likelihood = lda.score(X)

    # Total word count across all documents
    total_word_count = X.sum()

    perplexity = np.exp(-total_log_likelihood / total_word_count)
    return perplexity


# ── 2. Topic Coherence (NPMI) ──────────────────────────────────────────────
def compute_lda_npmi(lda, X_train_dense, vocab, top_n=10):
    """
    Compute NPMI topic coherence for sklearn LDA.
    Uses same method as evaluate.py for fair comparison with AVITM.

    Parameters
    ----------
    lda           : fitted LatentDirichletAllocation
    X_train_dense : np.ndarray (n_docs, vocab_size) — dense BOW matrix
    vocab         : np.ndarray of shape (vocab_size,)
    top_n         : int

    Returns
    -------
    avg_npmi    : float
    topic_npmis : list of float
    topic_words : list of list of str
    """
    n_docs     = X_train_dense.shape[0]
    vocab_size = len(vocab)

    # ── Word co-occurrence statistics ─────────────────────────────────────
    print("  Computing word co-occurrence statistics...")
    binary = (X_train_dense > 0).astype(float)      # (n_docs, vocab_size)
    word_counts = binary.mean(axis=0)                # p(w_i)

    # ── Extract top words per topic ────────────────────────────────────────
    topic_words = []
    for topic in lda.components_:
        top_indices = topic.argsort()[-top_n:][::-1]
        topic_words.append([vocab[i] for i in top_indices])

    # ── Compute NPMI per topic ─────────────────────────────────────────────
    word_to_idx  = {w: i for i, w in enumerate(vocab)}
    topic_npmis  = []

    for k, words in enumerate(topic_words):
        top_indices = [word_to_idx[w] for w in words if w in word_to_idx]

        if len(top_indices) < 2:
            topic_npmis.append(0.0)
            continue

        pair_npmis = []
        for i, j in combinations(top_indices, 2):
            p_i      = word_counts[i]
            p_j      = word_counts[j]
            co_occur = (binary[:, i] * binary[:, j]).mean()

            if co_occur < 1e-12 or p_i < 1e-12 or p_j < 1e-12:
                continue

            pmi  = np.log(co_occur / (p_i * p_j))
            npmi = pmi / (-np.log(co_occur))
            pair_npmis.append(npmi)

        topic_npmi = np.mean(pair_npmis) if pair_npmis else 0.0
        topic_npmis.append(topic_npmi)

    avg_npmi = float(np.mean(topic_npmis))
    return avg_npmi, topic_npmis, topic_words


# ── Main ───────────────────────────────────────────────────────────────────
def run_lda_baseline():
    """
    Full LDA baseline pipeline:
        1. Load data (reuse same DataLoaders as AVITM for consistency)
        2. Train sklearn LDA
        3. Evaluate perplexity and NPMI
        4. Save results
    """
    save_dir = Path("checkpoints")
    save_dir.mkdir(exist_ok=True)

    # ── Load data ──────────────────────────────────────────────────────────
    print("Loading data...")
    train_loader, test_loader, vocab = get_dataloaders()

    # Reconstruct dense numpy matrices from DataLoader
    # (sklearn needs numpy arrays, not PyTorch DataLoaders)
    print("Reconstructing numpy matrices...")
    X_train = np.vstack([batch.numpy() for batch in train_loader])
    X_test  = np.vstack([batch.numpy() for batch in test_loader])
    print(f"  Train: {X_train.shape}, Test: {X_test.shape}")

    # Convert to sparse for sklearn efficiency
    from scipy.sparse import csr_matrix
    X_train_sparse = csr_matrix(X_train)
    X_test_sparse  = csr_matrix(X_test)

    # ── Train LDA ──────────────────────────────────────────────────────────
    print(f"\nTraining sklearn LDA (K={N_TOPICS}, max_iter={MAX_ITER})...")
    lda = LatentDirichletAllocation(
        n_components=N_TOPICS,
        max_iter=MAX_ITER,
        learning_method='online',     # same as Hoffman et al. 2010 baseline
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    lda.fit(X_train_sparse)
    print("  Done.")

    # ── Perplexity ─────────────────────────────────────────────────────────
    print("\nComputing perplexity...")
    train_ppl = compute_lda_perplexity(lda, X_train_sparse, len(vocab))
    test_ppl  = compute_lda_perplexity(lda, X_test_sparse,  len(vocab))
    print(f"  Train perplexity : {train_ppl:.1f}")
    print(f"  Test  perplexity : {test_ppl:.1f}")

    # ── NPMI ──────────────────────────────────────────────────────────────
    print("\nComputing NPMI...")
    avg_npmi, topic_npmis, topic_words = compute_lda_npmi(
        lda, X_train, vocab, top_n=10
    )
    print(f"  Average NPMI : {avg_npmi:.4f}")

    # ── Print topics ───────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  LDA Baseline Topics (top 10 words)")
    print(f"{'='*60}")
    for k, words in enumerate(topic_words[:10]):   # show first 10
        print(f"  Topic {k+1:2d} [NPMI: {topic_npmis[k]:.3f}]: {', '.join(words)}")
    print(f"  ... ({N_TOPICS - 10} more topics)")
    print(f"{'='*60}")

    # ── Save results ───────────────────────────────────────────────────────
    results = {
        "model"        : "LDA (sklearn online VI)",
        "n_topics"     : N_TOPICS,
        "train_ppl"    : float(train_ppl),
        "test_ppl"     : float(test_ppl),
        "avg_npmi"     : float(avg_npmi),
        "topic_npmis"  : [float(x) for x in topic_npmis],
        "topic_words"  : topic_words,
    }
    results_path = save_dir / "lda_baseline_results.json"
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {results_path}")

    return results


# ── Entry point ────────────────────────────────────────────────────────────
if __name__ == "__main__":
    results = run_lda_baseline()

    print(f"\n{'='*40}")
    print(f"  LDA Baseline Summary")
    print(f"{'='*40}")
    print(f"  Test Perplexity : {results['test_ppl']:.1f}")
    print(f"  Avg NPMI        : {results['avg_npmi']:.4f}")
    print(f"{'='*40}")

Loading data...
Loading 20 Newsgroups...
Saving raw data locally...
  Saved to data/train_raw_data.json and data/test_raw_data.json
  Train docs : 11314
  Test docs  : 7532
Fitting CountVectorizer...
  Vocabulary size : 2000
  Train matrix    : (11314, 2000)
  Test matrix     : (7532, 2000)
  Vocab saved to data\vocab.txt
  After filtering empty docs:
    Train : 10938 docs
    Test  : 7249 docs

DataLoaders ready.
  Train batches : 54
  Test batches  : 37
Reconstructing numpy matrices...
  Train: (10800, 2000), Test: (7249, 2000)

Training sklearn LDA (K=50, max_iter=100)...
  Done.

Computing perplexity...
  Train perplexity : 1129.8
  Test  perplexity : 1309.2

Computing NPMI...
  Computing word co-occurrence statistics...
  Average NPMI : 0.2785

  LDA Baseline Topics (top 10 words)
  Topic  1 [NPMI: 0.133]: zone, zip, young, york, yesterday, yes, years, year, yeah, xterm
  Topic  2 [NPMI: 0.350]: file, files, program, entry, section, directory, format, jpeg, use, bits
  Topic  3 [